In [ ]:
import duckdb
import chromadb
import numpy as np
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Path resolution
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

CORPUS_PATH = PROJECT_ROOT / "data" / "processed" / "corpus.txt"

# Load corpus
with open(CORPUS_PATH, encoding="utf-8") as f:
    all_sentences = [line.strip() for line in f.readlines()]

# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

print(f"Corpus: {len(all_sentences)} sentences")
print(f"Model: {model.get_embedding_dimension()}d embeddings")

In [ ]:
# Initialize ChromaDB with persistent storage
chroma_client = chromadb.PersistentClient(
    path=str(PROJECT_ROOT / "chroma_db")
)

# Delete collection if it exists (clean slate)
try:
    chroma_client.delete_collection("f1_corpus")
    print("Deleted existing collection")
except:
    pass

collection = chroma_client.create_collection(
    name="f1_corpus",
    metadata={"hnsw:space": "cosine"}
)

print(f"Collection created: {collection.name}")
print("Embedding and indexing all sentences (this takes 2-3 minutes)...")

# Index in batches of 500
batch_size = 500
for i in range(0, len(all_sentences), batch_size):
    batch = all_sentences[i:i+batch_size]
    embeddings = model.encode(batch).tolist()
    ids = [str(j) for j in range(i, i+len(batch))]
    
    collection.add(
        documents=batch,
        embeddings=embeddings,
        ids=ids
    )
    print(f"  Indexed {min(i+batch_size, len(all_sentences))}/{len(all_sentences)}")

print(f"\nDone. Total indexed: {collection.count()}")

In [ ]:
def retrieve(query: str, top_k: int = 5) -> list[dict]:
    """
    Embed a query and find the top_k most similar sentences in ChromaDB.
    This is the R in RAG.
    """
    query_embedding = model.encode([query]).tolist()
    
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    
    retrieved = []
    for doc, distance in zip(results["documents"][0], results["distances"][0]):
        retrieved.append({
            "text": doc,
            "score": 1 - distance  # ChromaDB cosine returns distance, convert to similarity
        })
    
    return retrieved

# Test it
results = retrieve("Who won the most races for Mercedes?")
print("Query: 'Who won the most races for Mercedes?'\n")
for r in results:
    print(f"[{r['score']:.4f}] {r['text']}")

In [ ]:
import os
import ollama

OLLAMA_HOST = os.environ.get("F1_OLLAMA_URL", "http://localhost:11434")
client = ollama.Client(host=OLLAMA_HOST)

def generate(query: str, context: list[dict]) -> str:
    """
    Given a query and retrieved context, ask Ollama to generate an answer.
    This is the G in RAG.
    """
    context_text = "\n".join([f"- {r['text']}" for r in context])
    
    prompt = f"""You are an F1 expert assistant. Answer the question using ONLY the context below.
Synthesize the context into a direct, confident answer.
If asked about performance across multiple races, summarize the pattern you see.
If the context truly doesn't contain the answer, say so in one sentence.

Context:
{context_text}


Question: {query}

Answer:"""

    response = client.chat(
        model="llama3.2:3b",
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response["message"]["content"]

# Test the generator alone first
test_context = [{"text": "In the 2020 Bahrain Grand Prix, Lewis Hamilton finished 1st (winner) starting from position 1, scoring 25 points for Mercedes."}]
print(generate("Who won the 2020 Bahrain Grand Prix?", test_context))

In [ ]:
def ask(query: str, top_k: int = 5, verbose: bool = True) -> str:
    """
    Full RAG pipeline:
    1. Retrieve relevant sentences from ChromaDB
    2. Generate answer using Ollama with those sentences as context
    """
    # Step 1: Retrieve
    context = retrieve(query, top_k=top_k)
    
    if verbose:
        print(f"Query: {query}")
        print(f"\nRetrieved context:")
        for r in context:
            print(f"  [{r['score']:.4f}] {r['text']}")
        print()
    
    # Step 2: Generate
    answer = generate(query, context)
    
    if verbose:
        print(f"Answer:\n{answer}")
    
    return answer

In [ ]:
questions = [
    "Who won the 2021 Abu Dhabi Grand Prix?",
    "Which constructor dominated the 2014 season?",
    "How did Max Verstappen perform at Monaco?",
    "Who had the most wins for Ferrari in the 2000s?",
]

for q in questions:
    print("="*60)
    ask(q)
    print()

In [ ]:
# Ask something the corpus genuinely can't answer
hard_questions = [
    "What was the weather like at the 2019 German Grand Prix?",
    "How many pit stops did Hamilton make at Silverstone 2023?",
    "What is Verstappen's favourite circuit?",
]

print("Testing questions the corpus can't fully answer:\n")
for q in hard_questions:
    print("="*60)
    ask(q)
    print()

In [ ]:
## What We Just Built

The full RAG pipeline in ~60 lines of code:

- **Retrieve**: embed query → cosine search ChromaDB → top-5 sentences
- **Generate**: stuff context into prompt → Ollama generates grounded answer

### Failure modes observed:
1. **Retrieval failure** — wrong sentences returned, answer is hallucinated
2. **Generation failure** — right sentences returned, LLM reasons incorrectly
3. **Coverage gap** — question needs data not in corpus (weather, pit stops)

These are the three failure modes of every RAG system.
Notebooks 5 and 6 address failure modes 1 and 3.